# Telecom Customer Analytics — Public Portfolio Version



## Configurations

In [ ]:
#----- Import libraries -----
import pandas as pd
import time
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, silhouette_samples
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import ConfusionMatrixDisplay

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.base import clone

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)


In [ ]:
# ----- Random seed (reproductibilité) -----
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# ----- Style de visualisation -----
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 11
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)


In [ ]:
#----- Paramètres globaux du projet -----
import os

churn_threshold = os.getenv("CHURN_THRESHOLD_DAYS")
if churn_threshold is None:
    raise RuntimeError(
        "Set CHURN_THRESHOLD_DAYS to a public/approved value before running the notebook."
    )

CONFIG = {
    "data_path": os.getenv("TELECOM_DATA_PATH", "../data/customer_transactions.csv"),
    "reference_date": None,
    "churn_threshold_days": int(churn_threshold),
    "rfm_bins": 5,
    "test_size": 0.2,
    "nouveau_client_mois": 3,
}
CONFIG


## Data loading

In [ ]:
#----- Chargement des données -----
df = pd.read_csv(CONFIG["data_path"])


In [ ]:
#----- Dataset overview -----
df.head()


In [ ]:
# ----- Dimensions -----
print(f"Nombre de lignes    : {df.shape[0]}")
print(f"Nombre de colonnes  : {df.shape[1]}")


In [ ]:
#------ Dataset overview -----
df.describe()


In [ ]:
#----- Dataset overview 2 -----
df.describe(include="all").T


In [ ]:
# ----- Types de données -----
print(df.dtypes)


# Data preprocessing

## 1. Missing values

In [ ]:
#----- Missing values -----
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_pct", ascending=False)
missing_summary


In [ ]:
#------ Handling missing values : life_time -----
df["activation_date"] = pd.to_datetime(
    df["activation_date"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)
print(f"Type de la colonne 'activation_date' : {df['activation_date'].dtype}")

# Date de reference unique pour tout le projet (meme date que celle reutilisee plus loin
# pour le calcul du RFM) : on la calcule ici pour rester coherent d'un bout a l'autre du
# notebook, plutot que d'utiliser une date codee en dur qui pourrait diverger du RFM.
df["date_achat"] = pd.to_datetime(df["date_achat"], errors="coerce")
REFERENCE_DATE = (
    df["date_achat"].max() + pd.Timedelta(days=1)
    if CONFIG["reference_date"] is None
    else pd.Timestamp(CONFIG["reference_date"])
)
print(f"Date de reference unique du projet (utilisee pour life_time et pour le RFM) : {REFERENCE_DATE}")

mask = df["life_time"].isna()

df.loc[mask, "life_time"] = (
    (REFERENCE_DATE.year - df.loc[mask, "activation_date"].dt.year) * 12
    + (REFERENCE_DATE.month - df.loc[mask, "activation_date"].dt.month)
)

print(f"Nombre de valeurs manquantes restantes dans 'life_time' : {df['life_time'].isnull().sum()}")


## 2. Duplicate records

In [ ]:
#----- Detect duplicates -----
duplicates = df.duplicated().sum()
print(f"Nombre de doublons : {duplicates}")


In [ ]:
#----- Remove duplicates -----
if df.duplicated().sum() > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicates found.")


## 3. Data types

In [ ]:
#----- Data types (avant conversion) -----
print(df.dtypes)


In [ ]:
#------ Convert dates -----
date_columns = [
    "modification_date",
    "deactivation_date",
    "date_achat"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")


In [ ]:
#------ Convert categorical columns -----
categorical_columns = [
    "msisdn_crypte",
    "prgname",
    "offer_name",
    "code_option",
    "option_type",
    "lib_options",
    "prgcode",
    "tmcode"
]

for col in categorical_columns:
    df[col] = df[col].astype("category")


In [ ]:
#----- Convert numeric columns -----
numeric_columns = [
    "life_time",
    "montant"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)


## 4. Feature selection

_(nothing yet.)_

# EDA

## Dataset summary

In [ ]:
print(f"Shape (lignes, colonnes)      : {df.shape}")
print(f"Nombre de variables            : {df.shape[1]}")
print()
print("Numerical variables:", len(df.select_dtypes(include="number").columns))
print("Categorical variables:", len(df.select_dtypes(include="category").columns))
print("Date variables:", len(df.select_dtypes(include="datetime").columns))

# Number of categories
categorical_cols = df.select_dtypes(include=["category", "object"]).columns

summary_cat = pd.DataFrame({
    "Variable": categorical_cols,
    "Number of Categories": [df[col].nunique() for col in categorical_cols]
})

summary_cat


## Visualisation

In [ ]:
#----- Visualisation des valeurs manquantes -----
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis", yticklabels=False)
plt.title("Carte des valeurs manquantes")
plt.tight_layout()
plt.show()


In [ ]:
#----- Distribution des variables numériques -----
num_cols = ["montant", "life_time"]

fig, axes = plt.subplots(len(num_cols), 2, figsize=(14, 5 * len(num_cols)))

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i, 0], color="darkorange")
    axes[i, 0].set_title(f"Distribution de {col}")

    sns.boxplot(x=df[col], ax=axes[i, 1], color="darkorange")
    axes[i, 1].set_title(f"Boxplot de {col}")

plt.tight_layout()
plt.show()


In [ ]:
#----- Top catégories des variables catégorielles -----
cat_cols_to_plot = ["offer_name", "prgname", "option_type"]
top_n = 10

for col in cat_cols_to_plot:
    plt.figure(figsize=(10, 6))
    top_values = df[col].value_counts().nlargest(top_n)
    sns.barplot(x=top_values.values, y=top_values.index, orient="h")
    plt.title(f"Top {top_n} - {col}")
    plt.xlabel("Nombre de clients")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


In [ ]:
#----- Revenu total par offre (top 10) -----
offer_rev = (
    df.groupby("offer_name", observed=True)["montant"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
offer_rev.head(10).plot(kind="bar", color="darkorange")
plt.title("Top 10 des offres par montant total")
plt.xlabel("Offre")
plt.ylabel("Montant total")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
#----- Montant vs life_time -----
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="life_time", y="montant", alpha=0.3, color="darkorange")
plt.title("Montant en fonction de l'ancienneté (life_time)")
plt.tight_layout()
plt.show()


In [ ]:
#----- Corrélation des variables numériques -----
num_df = df.select_dtypes(include="number")

plt.figure(figsize=(8, 6))
sns.heatmap(num_df.corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()


In [ ]:
#----- Évolution des activations dans le temps -----
activations_par_mois = (
    df["activation_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(14, 6))
activations_par_mois.plot(kind="line", marker="o", color="darkorange")
plt.title("Nombre d'activations par mois")
plt.xlabel("Mois")
plt.ylabel("Nombre de clients activés")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
#----- Évolution des modifications dans le temps -----
modifications_par_mois = (
    df["modification_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(14, 6))
modifications_par_mois.plot(kind="line", marker="o", color="darkorange")
plt.title("Nombre de modifications par mois")
plt.xlabel("Mois")
plt.ylabel("Nombre de clients modifiés")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
#----- Évolution des désactivations dans le temps -----
# Attention : deactivation_date a un fort taux de valeurs manquantes (~94%),
# ce graphique ne porte donc que sur un sous-échantillon des clients.
deactivations_par_mois = (
    df["deactivation_date"]
    .dropna()
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

if deactivations_par_mois.empty:
    print("Aucune désactivation renseignée dans les données : graphique ignoré.")
else:
    plt.figure(figsize=(14, 6))
    deactivations_par_mois.plot(kind="line", marker="o", color="crimson")
    plt.title("Nombre de désactivations par mois")
    plt.xlabel("Mois")
    plt.ylabel("Nombre de clients désactivés")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# RFM par client

In [ ]:
#----- Date de reference -----

reference_date = REFERENCE_DATE
print(f"Date de reference : {reference_date}")


In [ ]:
#----- Nombre de clients uniques -----
nb_clients = df["msisdn_crypte"].nunique()
print(f"Nombre de clients uniques : {nb_clients}")

In [ ]:

rfm_client = df.groupby("msisdn_crypte", observed=True).agg(
    Recency=("date_achat", lambda x: (reference_date - x.max()).days),
    Frequency=("date_achat", "count"),
    Monetary=("montant", "sum")
).reset_index()

rfm_client = rfm_client.sort_values("Monetary", ascending=False)
rfm_client


In [ ]:
main_offer = (
    df.groupby("msisdn_crypte", observed=True)["offer_name"]
    .agg(lambda x: x.mode()[0])
    .rename("Main_Offer")
)

nb_offers_distinctes = (
    df.groupby("msisdn_crypte", observed=True)["offer_name"]
    .nunique()
    .rename("Nb_Offers_Distinctes")
)

rfm_client = rfm_client.merge(main_offer, on="msisdn_crypte")
rfm_client = rfm_client.merge(nb_offers_distinctes, on="msisdn_crypte")


activation_date_client = (
    df.groupby("msisdn_crypte", observed=True)["activation_date"]
    .min()
    .rename("activation_date")
)
rfm_client = rfm_client.merge(activation_date_client, on="msisdn_crypte", how="left")
rfm_client["Tenure_Months"] = (
    (reference_date - rfm_client["activation_date"]).dt.days / 30
).round(1)

rfm_client


In [ ]:
rfm_client["Nb_Offers_Distinctes"].describe()

## Scores

In [ ]:
bins = CONFIG["rfm_bins"]

rfm_client["R_score"] = pd.qcut(
    rfm_client["Recency"].rank(method="first"), bins, labels=range(bins, 0, -1)
).astype(int)

rfm_client["F_score"] = pd.qcut(
    rfm_client["Frequency"].rank(method="first"), bins, labels=range(1, bins + 1)
).astype(int)

rfm_client["M_score"] = pd.qcut(
    rfm_client["Monetary"].rank(method="first"), bins, labels=range(1, bins + 1)
).astype(int)

rfm_client["RFM_Score"] = (
    rfm_client["R_score"].astype(str)
    + rfm_client["F_score"].astype(str)
    + rfm_client["M_score"].astype(str)
)
rfm_client["RFM_Total"] = rfm_client[["R_score", "F_score", "M_score"]].sum(axis=1)

rfm_client.sort_values("RFM_Total", ascending=False)


## Segments

In [ ]:
def segment_client(score):
    if score >= 13:
        return "Champion"
    elif score >= 10:
        return "Client fidèle"
    elif score >= 7:
        return "Client moyen"
    else:
        return "Client à risque"

rfm_client["Segment"] = rfm_client["RFM_Total"].apply(segment_client)
rfm_client[["msisdn_crypte", "Main_Offer", "Recency", "Frequency", "Monetary", "RFM_Total", "RFM_Score", "Segment"]]


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(
    data=rfm_client, y="Segment",
    order=rfm_client["Segment"].value_counts().index
)
plt.title("Nombre de clients par segment RFM")
plt.xlabel("Nombre de clients")
plt.tight_layout()
plt.show()


In [ ]:
top_offers_list = rfm_client["Main_Offer"].value_counts().nlargest(10).index
cross_tab = pd.crosstab(
    rfm_client[rfm_client["Main_Offer"].isin(top_offers_list)]["Main_Offer"],
    rfm_client[rfm_client["Main_Offer"].isin(top_offers_list)]["Segment"]
)

plt.figure(figsize=(10, 8))
sns.heatmap(cross_tab, annot=True, fmt="d", cmap="YlOrRd")
plt.title("Segments RFM par offre principale (top 10 des offres)")
plt.tight_layout()
plt.show()


**Ce qu'on observe :** la repartition des clients entre `Champion`, `Client fidele`, `Client
moyen` et `Client a risque` n'est pas uniforme, et certaines offres concentrent davantage de
clients `Champion` ou, a l'inverse, de clients `a risque` que d'autres.

**What it means  :** le score RFM simple suffit deja a reperer les offres qui retiennent le
mieux leurs clients, et celles qui en perdent le plus.




## Distributions de RFM


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(rfm_client["Recency"], bins=30, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution de la Récence (jours)")
axes[0].set_xlabel("Recency")

sns.histplot(rfm_client["Frequency"], bins=30, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Distribution de la Fréquence")
axes[1].set_xlabel("Frequency")

sns.histplot(rfm_client["Monetary"], bins=30, kde=True, ax=axes[2], color="seagreen")
axes[2].set_title("Distribution du Montant total dépensé")
axes[2].set_xlabel("Monetary")

plt.tight_layout()
plt.show()

rfm_client[["Recency", "Frequency", "Monetary"]].describe()

**Ce qu'on observe :** les distributions de Recency, Frequency et Monetary sont typiquement
asymetriques (beaucoup de clients peu actifs/peu depensiers, une minorite tres active).

**What it means :** c'est pour cela que les scores RFM sont calcules par quantiles (bins
egaux en nombre de clients) plutot que par seuils fixes : une echelle lineaire serait dominee
par les quelques clients extremes.



## Bornes des quantiles utilisés pour les scores R, F, M



In [ ]:
for col, score_col in zip(["Recency", "Frequency", "Monetary"], ["R_score", "F_score", "M_score"]):
    bornes = rfm_client.groupby(score_col)[col].agg(["min", "max", "mean", "count"])
    print(f"----- Bornes du {score_col} (basé sur {col}) -----")
    print(bornes, "\n")

## RFM avancée



In [ ]:
def segment_comportemental(row):
    """Segment basé uniquement sur R (récence) et F (fréquence)."""
    r, f = row["R_score"], row["F_score"]

    if r >= 4 and f >= 4:
        return "Champions"
    elif r >= 4 and f >= 2:
        return "Clients fidèles"
    elif r >= 4:
        return "Clients prometteurs"      
    elif r == 3 and f >= 3:
        return "Clients fidèles"
    elif r == 3:
        return "Clients à développer"       
    elif r <= 2 and f >= 4:
        return "Ne doit pas les perdre"     
    elif r <= 2 and f == 3:
        return "Clients à risque"
    else:
        return "Clients hibernants"

def segment_avance(row):
    tenure = row.get("Tenure_Months")

    # Priorité absolue : ancienneté réelle du compte (déjà en place, on garde)
    if tenure is not None and tenure <= CONFIG["nouveau_client_mois"]:
        return "Nouveaux clients"

    return segment_comportemental(row)


rfm_client["Valeur"] = np.where(rfm_client["M_score"] >= 4, "Haute valeur", "Valeur standard")

## Profil détaillé de chaque segment

In [ ]:
rfm_client["Segment_Avance"] = rfm_client.apply(segment_avance, axis=1)

profil_segments = rfm_client.groupby("Segment_Avance").agg(
    Nb_clients=("msisdn_crypte", "count"),
    Recency_moy=("Recency", "mean"),
    Frequency_moy=("Frequency", "mean"),
    Monetary_moy=("Monetary", "mean"),
    Revenu_total=("Monetary", "sum"),
).sort_values("Revenu_total", ascending=False)

profil_segments["% Clients"] = (profil_segments["Nb_clients"] / profil_segments["Nb_clients"].sum() * 100).round(1)
profil_segments["% Revenu"] = (profil_segments["Revenu_total"] / profil_segments["Revenu_total"].sum() * 100).round(1)

profil_segments.round(2)


##  nombre de clients vs chiffre d'affaires


In [ ]:
order = profil_segments.index

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=profil_segments.loc[order, "% Clients"], y=order, hue=order, ax=axes[0], palette="Blues_r", legend=False)
axes[0].set_title("% de clients par segment")
axes[0].set_xlabel("% Clients")

sns.barplot(x=profil_segments.loc[order, "% Revenu"], y=order, hue=order, ax=axes[1], palette="Greens_r", legend=False)
axes[1].set_title("% du chiffre d'affaires par segment")
axes[1].set_xlabel("% Revenu")

plt.tight_layout()
plt.show()

**Ce qu'on observe :** la part de clients d'un segment et sa part de chiffre d'affaires ne
coincident generalement pas — un petit segment (ex. `Champions`) peut peser bien plus lourd en
revenu que sa taille ne le laisse penser, et inversement pour les segments a faible valeur.

**What it means:** tous les clients n'ont pas la meme valeur pour l'entreprise, meme si
le service prepaye est le meme pour tous.




## Carte de chaleur



In [ ]:
pivot_rfm = rfm_client.pivot_table(
    index="F_score", columns="R_score", values="Monetary", aggfunc="mean"
).sort_index(ascending=False)

plt.figure(figsize=(8, 6))
sns.heatmap(pivot_rfm, annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("Montant moyen dépensé selon Récence x Fréquence")
plt.xlabel("R_score (5 = très récent)")
plt.ylabel("F_score (5 = très fréquent)")
plt.tight_layout()
plt.show()

## Top clients par valeur (Champions)



In [ ]:
top_champions = (
    rfm_client[rfm_client["Segment_Avance"] == "Champions"]
    .sort_values("Monetary", ascending=False)
    .head(10)
)
top_champions[["msisdn_crypte", "Main_Offer", "Recency", "Frequency", "Monetary", "RFM_Total"]]

## Clients à risque 



In [ ]:
clients_a_risque = rfm_client[rfm_client["Segment_Avance"] == "Clients à risque"]

print(f"Nombre de clients à risque        : {len(clients_a_risque)}")
print(f"Chiffre d'affaires historique concerné : {clients_a_risque['Monetary'].sum():,.0f}")

clients_a_risque.sort_values("Monetary", ascending=False).head(10)[
    ["msisdn_crypte", "Main_Offer", "Recency", "Frequency", "Monetary", "RFM_Total"]
]

**Ce qu'on observe :** le segment `Clients a risque` regroupe des clients avec une recence et
une frequence degradees ; certains representaient un chiffre d'affaires historique non
negligeable.

**What it means:** ce sont des clients qui ont probablement deja commence a se detourner
de l'offre, avant meme de basculer dans la definition du churn utilisee pour l'analyse (Recency > seuil configure).



## Évolution mensuelle du chiffre d'affaires par segment RFM



In [ ]:
df_seg = df.merge(rfm_client[["msisdn_crypte", "Segment_Avance"]], on="msisdn_crypte", how="left")
df_seg["mois_achat"] = df_seg["date_achat"].dt.to_period("M").dt.to_timestamp()

evolution_ca = (
    df_seg.groupby(["mois_achat", "Segment_Avance"], observed=True)["montant"]
    .sum()
    .reset_index()
)

plt.figure(figsize=(14, 6))
sns.lineplot(data=evolution_ca, x="mois_achat", y="montant", hue="Segment_Avance", marker="o")
plt.title("Évolution mensuelle du chiffre d'affaires par segment RFM")
plt.xlabel("Mois")
plt.ylabel("Chiffre d'affaires")
plt.xticks(rotation=45)
plt.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Évolution mensuelle du nombre de clients actifs par segment



In [ ]:
evolution_clients = (
    df_seg.groupby(["mois_achat", "Segment_Avance"], observed=True)["msisdn_crypte"]
    .nunique()
    .reset_index(name="Nb_clients_actifs")
)

plt.figure(figsize=(14, 6))
sns.lineplot(data=evolution_clients, x="mois_achat", y="Nb_clients_actifs", hue="Segment_Avance", marker="o")
plt.title("Évolution mensuelle du nombre de clients actifs par segment")
plt.xlabel("Mois")
plt.ylabel("Nombre de clients actifs")
plt.xticks(rotation=45)
plt.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Courbe de Lorenz 


In [ ]:
rfm_sorted = rfm_client.sort_values("Monetary", ascending=False).reset_index(drop=True)
rfm_sorted["cum_clients_pct"] = (rfm_sorted.index + 1) / len(rfm_sorted) * 100
rfm_sorted["cum_revenue_pct"] = rfm_sorted["Monetary"].cumsum() / rfm_sorted["Monetary"].sum() * 100

idx_20 = max(int(len(rfm_sorted) * 0.2) - 1, 0)
part_revenu_top20 = rfm_sorted.loc[idx_20, "cum_revenue_pct"]

plt.figure(figsize=(8, 6))
plt.plot(rfm_sorted["cum_clients_pct"], rfm_sorted["cum_revenue_pct"], color="crimson", linewidth=2, label="Courbe de Lorenz")
plt.plot([0, 100], [0, 100], linestyle="--", color="grey", label="Répartition égalitaire")
plt.scatter([20], [part_revenu_top20], color="black", zorder=5)
plt.annotate(
    f"Top 20% clients\n= {part_revenu_top20:.0f}% du CA",
    xy=(20, part_revenu_top20), xytext=(38, part_revenu_top20 - 20),
    arrowprops=dict(arrowstyle="->")
)

plt.title("Courbe de Lorenz — concentration du chiffre d'affaires")
plt.xlabel("% cumulé de clients (triés par valeur décroissante)")
plt.ylabel("% cumulé du chiffre d'affaires")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Les 20% de clients les plus rentables génèrent {part_revenu_top20:.1f}% du chiffre d\'affaires total.")

**Ce qu'on observe :** la courbe de Lorenz s'ecarte nettement de la diagonale d'egalite
parfaite.

**What it means :** le chiffre d'affaires est concentre sur une minorite de clients a
forte valeur, coherent avec la repartition observee par segment ci-dessus.




## Radar RFM par segment



In [ ]:
radar_df = rfm_client.groupby("Segment_Avance")[["R_score", "F_score", "M_score"]].mean()

categories = ["R_score", "F_score", "M_score"]
n_cat = len(categories)
angles = [n / float(n_cat) * 2 * np.pi for n in range(n_cat)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for segment in radar_df.index:
    values = radar_df.loc[segment].tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, marker="o", label=segment)
    ax.fill(angles, values, alpha=0.05)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(0, 5)
ax.set_title("Profil R-F-M moyen par segment", pad=30)
ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.1))
plt.tight_layout()
plt.show()

**Ce qu'on observe :** chaque segment comportemental a un profil R-F-M distinct sur le radar
(par exemple `Champions` haut sur les 3 axes, `Clients hibernants` bas sur les 3 axes).

**Ce que ca signifie :** les segments capturent bien des profils clients differents, pas
seulement une variation de bruit statistique.



## Courbes de densité 


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.kdeplot(data=rfm_client, x=col, hue="Segment_Avance", ax=ax, common_norm=False, fill=False, linewidth=1.5)
    ax.set_title(f"Densité de {col} par segment")

plt.tight_layout()
plt.show()

## RFM dynamique dans le temps 



In [ ]:
def compute_rfm_snapshot(data, snapshot_date, bins=CONFIG["rfm_bins"]):
    data_period = data[data["date_achat"] <= snapshot_date]

    if data_period["msisdn_crypte"].nunique() < bins:
        return None

    rfm_snap = data_period.groupby("msisdn_crypte", observed=True).agg(
        Recency=("date_achat", lambda x: (snapshot_date - x.max()).days),
        Frequency=("date_achat", "count"),
        Monetary=("montant", "sum"),
    ).reset_index()

    nb_offers_snap = (
        data_period.groupby("msisdn_crypte", observed=True)["offer_name"]
        .nunique()
        .rename("Nb_Offers_Distinctes")
    )
    rfm_snap = rfm_snap.merge(nb_offers_snap, on="msisdn_crypte")

    activation_date_snap = (
        data_period.groupby("msisdn_crypte", observed=True)["activation_date"]
        .min()
        .rename("activation_date")
    )
    rfm_snap = rfm_snap.merge(activation_date_snap, on="msisdn_crypte")
    rfm_snap["Tenure_Months"] = (
        (snapshot_date - rfm_snap["activation_date"]).dt.days / 30
    ).round(1)

    rfm_snap["R_score"] = pd.qcut(rfm_snap["Recency"].rank(method="first"), bins, labels=range(bins, 0, -1)).astype(int)
    rfm_snap["F_score"] = pd.qcut(rfm_snap["Frequency"].rank(method="first"), bins, labels=range(1, bins + 1)).astype(int)
    rfm_snap["M_score"] = pd.qcut(rfm_snap["Monetary"].rank(method="first"), bins, labels=range(1, bins + 1)).astype(int)

    rfm_snap["Segment_Avance"] = rfm_snap.apply(segment_avance, axis=1)
    rfm_snap["snapshot_date"] = snapshot_date
    return rfm_snap

In [ ]:
date_min = df["date_achat"].min()
date_max = df["date_achat"].max()

print(f"Historique disponible pour 'date_achat' : {date_min.date()} -> {date_max.date()} "
      f"({(date_max - date_min).days} jours)")

SNAPSHOT_FREQ = "1MS"  # snapshots mensuels (une valeur par mois)
snapshot_dates = pd.date_range(start=date_min, end=date_max, freq=SNAPSHOT_FREQ)

if len(snapshot_dates) < 2:
    print("Pas assez d'historique pour au moins 2 snapshots : "
          "les sections basées sur les snapshots (transition, modèle 'à risque', LTV) seront ignorées.")
    rfm_snapshots = {}
else:
    print(f"{len(snapshot_dates)} snapshots générés (fréquence : '{SNAPSHOT_FREQ}') : "
          f"{[d.date() for d in snapshot_dates]}")
    rfm_snapshots = {snap_date: compute_rfm_snapshot(df, snap_date) for snap_date in snapshot_dates}
    rfm_snapshots = {d: snap for d, snap in rfm_snapshots.items() if snap is not None}
    if len(rfm_snapshots) < 2:
        print("Après filtrage des snapshots trop petits, il en reste moins de 2 : "
              "les sections basées sur les snapshots seront ignorées.")

snapshots_disponibles = len(rfm_snapshots) >= 2


### Évolution du nombre de clients par segment au fil des périodes

In [ ]:
if snapshots_disponibles:
    evolution_segments = pd.concat(
        [snap.assign(period=snap_date) for snap_date, snap in rfm_snapshots.items()]
    )

    segment_counts_over_time = (
        evolution_segments.groupby(["period", "Segment_Avance"]).size().unstack(fill_value=0)
    )


    segment_pct_over_time = segment_counts_over_time.div(
        segment_counts_over_time.sum(axis=1), axis=0
    ) * 100

    segment_pct_over_time.plot(kind="line", marker="o", figsize=(14, 6))
    plt.title("Évolution de la répartition (%) des clients par segment RFM (snapshots)")
    plt.xlabel("Date du snapshot")
    plt.ylabel("% de clients")
    plt.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("Section ignorée : moins de 2 snapshots disponibles (historique de date_achat trop court).")

### Matrice de transition entre les deux dernières périodes



In [ ]:
if snapshots_disponibles:
    dates_sorted = sorted(rfm_snapshots.keys())
    date_t0, date_t1 = dates_sorted[-2], dates_sorted[-1]

    seg_t0 = rfm_snapshots[date_t0].set_index("msisdn_crypte")["Segment_Avance"]
    seg_t1 = rfm_snapshots[date_t1].set_index("msisdn_crypte")["Segment_Avance"]

    transition = pd.DataFrame({"Segment_avant": seg_t0, "Segment_apres": seg_t1}).dropna()

    transition_matrix = pd.crosstab(
        transition["Segment_avant"], transition["Segment_apres"], normalize="index"
    ) * 100

    plt.figure(figsize=(10, 8))
    sns.heatmap(transition_matrix, annot=True, fmt=".1f", cmap="RdYlGn_r")
    plt.title(f"Matrice de transition des segments : {date_t0.date()} -> {date_t1.date()} (%)")
    plt.xlabel("Segment à la période suivante")
    plt.ylabel("Segment à la période précédente")
    plt.tight_layout()
    plt.show()
else:
    date_t0, date_t1 = None, None
    transition = pd.DataFrame(columns=["Segment_avant", "Segment_apres"])
    print("Section ignorée : pas assez de snapshots pour calculer une matrice de transition.")


In [ ]:
if snapshots_disponibles:
    nouveaux_a_risque = transition[
        (transition["Segment_apres"] == "Clients à risque") & (transition["Segment_avant"] != "Clients à risque")
    ]

    print(f"Nombre de clients passés au segment 'À risque' entre les deux périodes : {len(nouveaux_a_risque)}")
    print("D'où venaient-ils ?")
    print(nouveaux_a_risque["Segment_avant"].value_counts())
else:
    print("Section ignorée : pas de transition disponible.")


## Score RFM pondere




In [ ]:

poids_rfm = {"R_score": 0.2, "F_score": 0.3, "M_score": 0.5}

rfm_client["RFM_Weighted"] = (
    rfm_client["R_score"] * poids_rfm["R_score"]
    + rfm_client["F_score"] * poids_rfm["F_score"]
    + rfm_client["M_score"] * poids_rfm["M_score"]
)

rfm_client[["msisdn_crypte", "RFM_Total", "RFM_Weighted"]].sort_values("RFM_Weighted", ascending=False).head(10)

In [ ]:
rfm_client["Rang_Total"] = rfm_client["RFM_Total"].rank(ascending=False, method="min")
rfm_client["Rang_Weighted"] = rfm_client["RFM_Weighted"].rank(ascending=False, method="min")
rfm_client["Ecart_Rang"] = (rfm_client["Rang_Total"] - rfm_client["Rang_Weighted"]).abs()

plt.figure(figsize=(7, 7))
plt.scatter(rfm_client["Rang_Total"], rfm_client["Rang_Weighted"], alpha=0.2, s=10)
lims = [1, len(rfm_client)]
plt.plot(lims, lims, "--", color="red", label="Classements identiques")
plt.xlabel("Rang (score simple R+F+M)")
plt.ylabel("Rang (score pondéré)")
plt.title("Score simple vs score pondéré : qui change de classement ?")
plt.legend()
plt.tight_layout()
plt.show()

print("Clients dont le classement change le plus selon la pondération :")
rfm_client.sort_values("Ecart_Rang", ascending=False)[
    ["msisdn_crypte", "R_score", "F_score", "M_score", "Rang_Total", "Rang_Weighted", "Ecart_Rang"]
].head(10)
#majority of these clients sont proches de la ligne rouge.
#Cela signifie que le RFM pondéré conserve globalement le même classement que le RFM classique.

## Precision : `Life_Time` vs `Tenure_Months`




In [ ]:

life_time_client = (
    df.groupby("msisdn_crypte", observed=True)["life_time"]
    .mean()
    .rename("Life_Time")
)
rfm_client = rfm_client.merge(life_time_client, on="msisdn_crypte", how="left")

rfm_client["Montant_Moyen_Par_Achat"] = rfm_client["Monetary"] / rfm_client["Frequency"].replace(0, 1)

tenure_non_nulle = rfm_client["Tenure_Months"].replace(0, np.nan).fillna(1)
rfm_client["Frequency_Par_Mois"] = rfm_client["Frequency"] / tenure_non_nulle
rfm_client["Monetary_Par_Mois"] = rfm_client["Monetary"] / tenure_non_nulle

rfm_client[["Life_Time", "Montant_Moyen_Par_Achat", "Frequency_Par_Mois", "Monetary_Par_Mois"]].describe()


# scaling  K-means

(avant le clustering) features


## Verification de la redondance des variables avant clustering

Plusieurs variables utilisees pour le clustering decrivent des notions proches (frequence
d'achat, montant, anciennete...). Avant de lancer le K-Means, on verifie leur correlation pour
savoir si certaines sont redondantes.


In [ ]:
redundancy_candidates = [
    "Frequency", "Frequency_Par_Mois", "Monetary", "Monetary_Par_Mois",
    "Tenure_Months", "Life_Time", "Montant_Moyen_Par_Achat",
]
redundancy_candidates = [c for c in redundancy_candidates if c in rfm_client.columns]

corr_redundancy = rfm_client[redundancy_candidates].corr().round(2)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_redundancy, annot=True, cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Correlation entre variables candidates au clustering")
plt.tight_layout()
plt.show()

corr_redundancy


In [ ]:

cluster_features = [
    "Recency", "Frequency", "Monetary",
    "Tenure_Months", "Nb_Offers_Distinctes",
    "Life_Time", "Montant_Moyen_Par_Achat", "Frequency_Par_Mois",
]
cluster_features = [c for c in cluster_features if c in rfm_client.columns]

rfm_for_cluster = rfm_client[cluster_features].copy()
rfm_for_cluster = rfm_for_cluster.fillna(rfm_for_cluster.median(numeric_only=True))

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_for_cluster)

rfm_scaled_df = pd.DataFrame(
    rfm_scaled,
    columns=[f"{col}_scaled" for col in cluster_features],
    index=rfm_client["msisdn_crypte"]
)

print(f"Features utilisees pour le clustering ({len(cluster_features)}) : {cluster_features}")
rfm_scaled_df.describe().loc[["mean", "std"]]


In [ ]:
print(rfm_snapshots.keys())

In [ ]:

pca = PCA(n_components=2, random_state=RANDOM_SEED)
pca_result = pca.fit_transform(rfm_scaled_df)

rfm_client["PC1"] = pca_result[:, 0]
rfm_client["PC2"] = pca_result[:, 1]

print("Variance expliquée :", pca.explained_variance_ratio_)
print("Total :", pca.explained_variance_ratio_.sum()) #86% dinfo est conserve


In [ ]:
inertias, silhouettes, davies_bouldins = [], [], []
k_range = range(2, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(rfm_scaled_df)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(rfm_scaled_df, labels))
    davies_bouldins.append(davies_bouldin_score(rfm_scaled_df, labels))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(k_range, inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertie")

axes[1].plot(k_range, silhouettes, marker="o", color="darkorange")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Score (plus haut = mieux)")

axes[2].plot(k_range, davies_bouldins, marker="o", color="crimson")
axes[2].set_title("Davies-Bouldin Index")
axes[2].set_xlabel("k")
axes[2].set_ylabel("Score (plus bas = mieux)")

plt.tight_layout()
plt.show()


In [ ]:

best_idx = int(np.argmax(silhouettes))
k_final = list(k_range)[best_idx]
print(f"k choisi automatiquement : {k_final} (silhouette = {silhouettes[best_idx]:.3f}, "
      f"davies-bouldin = {davies_bouldins[best_idx]:.3f})")

kmeans = KMeans(n_clusters=k_final, random_state=RANDOM_SEED, n_init=10)
rfm_client["Cluster_KMeans"] = kmeans.fit_predict(rfm_scaled_df)


labels_ref = rfm_client["Cluster_KMeans"].values
ari_scores = []
for seed in [0, 1, 7, 123, 2024]:
    labels_test = KMeans(n_clusters=k_final, random_state=seed, n_init=10).fit_predict(rfm_scaled_df)
    ari_scores.append(adjusted_rand_score(labels_ref, labels_test))

print(f"ARI moyen entre seeds : {np.mean(ari_scores):.3f} (proche de 1 = clusters stables, peu importe le seed)")

rfm_client.groupby("Cluster_KMeans")[cluster_features].mean()


**Ce qu'on observe :** le k retenu maximise le score de silhouette parmi les valeurs testees,
et l'ARI moyen entre plusieurs graines aleatoires est eleve.

**Ce que ca signifie :** la segmentation K-Means n'est pas un artefact du hasard
d'initialisation — elle est reproductible.


### Visualisations complementaires des segments K-Means


In [ ]:
# ----- Taille de chaque segment -----
plt.figure(figsize=(7, 5))
rfm_client["Cluster_KMeans"].value_counts().sort_index().plot(kind="bar", color="darkorange")
plt.title("Taille de chaque segment (K-Means)")
plt.xlabel("Cluster")
plt.ylabel("Nombre de clients")
plt.tight_layout()
plt.show()


In [ ]:
# ----- Projection PCA coloree par cluster -----
plt.figure(figsize=(8, 6))
sns.scatterplot(x="PC1", y="PC2", hue="Cluster_KMeans", data=rfm_client, palette="tab10")
plt.title(f"Clusters projetes sur les 2 premieres composantes PCA (k={k_final})")
plt.tight_layout()
plt.show()


In [ ]:
# ----- Distribution de chaque feature par cluster -----
n_feat = len(cluster_features)
n_cols = 4
n_rows = int(np.ceil(n_feat / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.array(axes).reshape(-1)
for ax, col in zip(axes, cluster_features):
    sns.boxplot(x="Cluster_KMeans", y=col, data=rfm_client, ax=ax)
    ax.set_title(col)
for ax in axes[n_feat:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# ----- Diagramme de silhouette par cluster (pas juste la moyenne) -----
import matplotlib.cm as cm

sil_values = silhouette_samples(rfm_scaled_df, rfm_client["Cluster_KMeans"])
y_lower = 10
fig, ax = plt.subplots(figsize=(8, 6))
for i in range(k_final):
    ith = sil_values[rfm_client["Cluster_KMeans"] == i]
    ith.sort()
    y_upper = y_lower + len(ith)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith, color=cm.tab10(i))
    ax.text(-0.02, y_lower + 0.5 * len(ith), str(i))
    y_lower = y_upper + 10
ax.axvline(x=silhouettes[best_idx], color="red", linestyle="--", label="Silhouette moyen")
ax.set_title("Diagramme de silhouette par cluster")
ax.set_xlabel("Coefficient de silhouette")
ax.set_ylabel("Clients (groupes par cluster)")
ax.legend()
plt.tight_layout()
plt.show()


**Ce qu'on observe :** le diagramme de silhouette par cluster permet de voir si un cluster en
particulier tire la qualite globale vers le bas (barres proches de 0, voire negatives), plutot
que de se fier uniquement a la moyenne globale.

**Ce que ca signifie :** si tous les clusters ont des coefficients de silhouette globalement
positifs et homogenes, la segmentation K-Means est fiable dans son ensemble.



In [ ]:

for eps_test in [0.1, 0.2, 0.3, 0.4, 0.5]: 
    db = DBSCAN(eps=eps_test, min_samples=10) #eps rayon de recherche
    
    labels = db.fit_predict(rfm_scaled_df )
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f"eps={eps_test} -> clusters: {n_clusters}, bruit: {n_noise}")

# Definition du churn

**Définition du churn (version publique) :** le seuil métier utilisé pendant le stage n'est pas publié dans ce dépôt.

Le notebook lit le seuil depuis la variable d'environnement `CHURN_THRESHOLD_DAYS`. Cela permet de reproduire la méthodologie sans exposer une règle métier interne.


In [ ]:

rfm_client["Churned"] = (rfm_client["Recency"] > CONFIG["churn_threshold_days"]).astype(int)
rfm_client["Churned"].value_counts(normalize=True)  

In [ ]:



feature_cols = [
    "Frequency", "Monetary", "Nb_Offers_Distinctes",
    "F_score", "M_score",
    "Life_Time", "Montant_Moyen_Par_Achat",
    "Tenure_Months", "Frequency_Par_Mois", "Monetary_Par_Mois",
]

X = rfm_client[feature_cols].copy()
X = X.fillna(X.median(numeric_only=True))

y = rfm_client["Churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"], random_state=RANDOM_SEED, stratify=y
)

print(f"Nombre de features : {X.shape[1]}")
X.head()


In [ ]:
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 2, 5],
}

start = time.time()
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED, class_weight="balanced"),
    param_grid=param_grid_rf,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)
grid_rf.fit(X_train, y_train)
temps_rf = time.time() - start

rf = grid_rf.best_estimator_
y_pred = rf.predict(X_test)
n_repeats = 50
start_pred = time.perf_counter()
for _ in range(n_repeats):
    rf.predict(X_test)
temps_pred_rf = (time.perf_counter() - start_pred) / n_repeats

print(f"Meilleurs paramètres (Random Forest) : {grid_rf.best_params_}")
print(f"Temps GridSearchCV (Random Forest) : {temps_rf:.1f}s")
print(f"Temps de prédiction (Random Forest) : {temps_pred_rf:.6f}s")
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot()
plt.title("Matrice de confusion - Random Forest (GridSearchCV)")
plt.show()


In [ ]:
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
}

start = time.time()
grid_xgb = GridSearchCV(
    XGBClassifier(random_state=RANDOM_SEED, scale_pos_weight=scale_pos, eval_metric="logloss"),
    param_grid=param_grid_xgb,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)
grid_xgb.fit(X_train, y_train)
temps_xgb = time.time() - start

xgb = grid_xgb.best_estimator_
y_pred_xgb = xgb.predict(X_test)
n_repeats = 50
start_pred = time.perf_counter()
for _ in range(n_repeats):
    xgb.predict(X_test)
temps_pred_xgb = (time.perf_counter() - start_pred) / n_repeats

print(f"Meilleurs paramètres (XGBoost) : {grid_xgb.best_params_}")
print(f"Temps GridSearchCV (XGBoost) : {temps_xgb:.1f}s")
print(f"Temps de prédiction (XGBoost) : {temps_pred_xgb:.6f}s")
print(classification_report(y_test, y_pred_xgb))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_xgb)).plot()
plt.title("Matrice de confusion - XGBoost (GridSearchCV)")
plt.show()


In [ ]:
param_grid_lgbm = {
    "n_estimators": [100, 200],
    "num_leaves": [15, 31, 63],
    "learning_rate": [0.05, 0.1],
}

start = time.time()
grid_lgbm = GridSearchCV(
    LGBMClassifier(random_state=RANDOM_SEED, class_weight="balanced"),
    param_grid=param_grid_lgbm,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)
grid_lgbm.fit(X_train, y_train)
temps_lgbm = time.time() - start

lgbm = grid_lgbm.best_estimator_
y_pred_lgbm = lgbm.predict(X_test)
n_repeats = 50
start_pred = time.perf_counter()
for _ in range(n_repeats):
    lgbm.predict(X_test)
temps_pred_lgbm = (time.perf_counter() - start_pred) / n_repeats

print(f"Meilleurs paramètres (LightGBM) : {grid_lgbm.best_params_}")
print(f"Temps GridSearchCV (LightGBM) : {temps_lgbm:.1f}s")
print(f"Temps de prédiction (LightGBM) : {temps_pred_lgbm:.6f}s")
print(classification_report(y_test, y_pred_lgbm))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lgbm)).plot()
plt.title("Matrice de confusion - LightGBM (GridSearchCV)")
plt.show()


In [ ]:
comparaison = pd.DataFrame({
    "Modele": ["Random Forest", "XGBoost", "LightGBM"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_lgbm),
    ],
    "F1-score (classe 1)": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_lgbm),
    ],
    "AUC-ROC": [
        roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, lgbm.predict_proba(X_test)[:, 1]),
    ],
    "Temps GridSearchCV (s)": [temps_rf, temps_xgb, temps_lgbm],
    "Temps prédiction (ms)": [
        temps_pred_rf * 1000,
        temps_pred_xgb * 1000,
        temps_pred_lgbm * 1000,
    ],
    "Temps prédiction (µs/client)": [
        temps_pred_rf / len(X_test) * 1e6,
        temps_pred_xgb / len(X_test) * 1e6,
        temps_pred_lgbm / len(X_test) * 1e6,
    ],
}).sort_values("F1-score (classe 1)", ascending=False)

print(f"Temps total (3 GridSearchCV) : {temps_rf + temps_xgb + temps_lgbm:.1f}s")
comparaison
#Area Under the Curve - Receiver Operating Characteristic
# . C'est une métrique qui mesure la capacité du modèle à séparer les deux classes (churn vs non-churn),
#  peu importe le seuil de décision choisi.

In [ ]:
# ----- Selection du modele gagnant (utilise pour la feature importance et le scoring churn) -----
modeles_disponibles = {"Random Forest": rf, "XGBoost": xgb, "LightGBM": lgbm}
meilleur_nom = comparaison.loc[comparaison["F1-score (classe 1)"].idxmax(), "Modele"]
meilleur_modele_fit = modeles_disponibles[meilleur_nom]
print(f"Modele retenu (meilleur F1-score) : {meilleur_nom}")


**Ce qu'on observe :** les 3 modeles obtiennent des scores relativement proches, mais un seul
maximise le F1-score sur la classe "churne", la metrique la plus pertinente ici (les classes
sont desequilibrees : l'accuracy seule serait trompeuse).

**Ce que ca signifie :** le modele affiche juste au-dessus ("Modele retenu") est celui qui
equilibre le mieux precision et rappel sur les clients reellement churnes.




In [ ]:
# ----- Verification explicite : l'importance ci-dessous provient bien du modele gagnant -----
assert meilleur_nom in modeles_disponibles, "meilleur_nom introuvable dans modeles_disponibles"
assert modeles_disponibles[meilleur_nom] is meilleur_modele_fit, (
    "Incoherence : meilleur_modele_fit ne correspond pas au modele gagnant selectionne."
)
print(f"Verification OK : l'importance des variables ci-dessous provient bien du modele "
      f"retenu ({meilleur_nom}), et non d'un modele fixe en dur.")

if hasattr(meilleur_modele_fit, "feature_importances_"):
    importances = pd.Series(
        meilleur_modele_fit.feature_importances_, index=X.columns
    ).sort_values(ascending=False)
else:
    importances = pd.Series(dtype=float)

plt.figure(figsize=(9, 6))
importances.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Feature Importance - {meilleur_nom}")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


**Ce qu'on observe :** les variables liees a la recence et a la frequence d'achat arrivent
generalement en tete de l'importance des variables, devant les variables purement monetaires.

**Ce que ca signifie :** un client churne surtout quand il cesse d'acheter regulierement, plus
que lorsqu'il achete moins cher en moyenne.




# Export du dataset final pour le dashboard

In [ ]:
# ----- Export du dataset final pour le dashboard -----
import os

# Dossier de sortie pour les fichiers du dashboard
OUTPUT_DIR = "dashboard_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Le modele gagnant (meilleur_nom / meilleur_modele_fit) a deja ete determine juste apres
# le tableau de comparaison des modeles, et est reutilise ici pour scorer tous les clients.
print(f"Modele retenu pour le scoring churn : {meilleur_nom}")

# ----- Probabilites de churn hors echantillon (out-of-fold) -----

cv_oof = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
estimateur_oof = clone(meilleur_modele_fit)
proba_oof = cross_val_predict(
    estimateur_oof, X, y, cv=cv_oof, method="predict_proba", n_jobs=-1
)
rfm_client["Churn_Proba"] = proba_oof[:, 1]

# Colonnes enrichies : Ive added les scores/segments RFM complets pour que le
# dashboard Streamlit puisse raconter toute l'histoire de l'analyse (segments
# simples ET comportementaux, score pondere, etc.), pas seulement le clustering
# et le churn.
export_cols = [
    "msisdn_crypte", "Recency", "Frequency", "Monetary",
    "Tenure_Months", "Nb_Offers_Distinctes", "Main_Offer",
    "R_score", "F_score", "M_score", "RFM_Score", "RFM_Total", "RFM_Weighted",
    "Segment", "Segment_Avance", "Valeur",
    "Life_Time", "Montant_Moyen_Par_Achat",
    "Frequency_Par_Mois", "Monetary_Par_Mois",
    "Cluster_KMeans", "Churned", "Churn_Proba",
    "PC1", "PC2",
]
export_cols = [c for c in export_cols if c in rfm_client.columns]

clients_export_path = os.path.join(OUTPUT_DIR, "clients_export.csv")
rfm_client[export_cols].to_csv(clients_export_path, index=False)
print(f"Exporte : {clients_export_path} ({rfm_client.shape[0]} lignes, {len(export_cols)} colonnes)")

# ----- Export des metriques du comparatif de modeles (pour le dashboard) -----
model_comparison_path = os.path.join(OUTPUT_DIR, "model_comparison.csv")
comparaison.to_csv(model_comparison_path, index=False)
print(f"Exporte : {model_comparison_path} ({comparaison.shape[0]} modeles)")

# ----- Export de l'importance des features du modele retenu (pour le dashboard) -----
feature_importance_path = os.path.join(OUTPUT_DIR, "feature_importance.csv")
importances.rename("Importance").rename_axis("Feature").reset_index().assign(
    Modele=meilleur_nom
).to_csv(feature_importance_path, index=False)
print(f"Exporte : {feature_importance_path} ({len(importances)} features, modele = {meilleur_nom})")